In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# -----------------------------
# 1. Функция для генерации Hamiltonian H2
# Коэффициенты взяты из минимальной модели STO-3G для разных длин связи
def h2_hamiltonian():
    # стандартный H2 в сокращённом виде
    pauli_list = [
        ("II", -1.052373245772859),
        ("ZI",  0.39793742484318045),
        ("IZ", -0.39793742484318045),
        ("ZZ", -0.01128010425623538),
        ("XX",  0.18093119978423156)
    ]
    return SparsePauliOp.from_list(pauli_list)

# -----------------------------
# 2. Ansatz для 2 кубитов
def ansatz(params):
    qc = QuantumCircuit(2)
    qc.ry(params[0], 0)
    qc.cx(0, 1)
    qc.ry(params[1], 1)
    return qc

# -----------------------------
# 3. Функция энергии
def energy(params, hamiltonian):
    sv = Statevector(ansatz(params))
    return np.real(sv.expectation_value(hamiltonian))

# -----------------------------
# 4. Функция VQE для одной длины связи
def vqe_for_distance(distance):
    # В этом упрощённом примере гамильтониан не меняется, но
    # в реальной модели он зависит от distance
    H = h2_hamiltonian()
    
    # Оптимизация
    init_params = np.random.rand(2)
    result = minimize(energy, init_params, args=(H,), method='COBYLA')
    
    return result.fun

# -----------------------------
# 5. Строим энергетическую кривую
distances = np.linspace(0.5, 2.0, 15)  # длина связи в ангстремах
energies = []

for d in distances:
    e = vqe_for_distance(d)
    energies.append(e)
    print(f"R = {d:.2f} Å, E = {e:.6f} Ha")

# -----------------------------
# 6. График
plt.plot(distances, energies, marker='o')
plt.xlabel("Длина связи H₂ (Å)")
plt.ylabel("Энергия (Ha)")
plt.title("Энергетическая кривая H₂ (VQE)")
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# from qiskit.circuit.library import TwoLocal
from qiskit.circuit.library import n_local

from qiskit_algorithms.minimum_eigensolvers import VQE
from qiskit_algorithms.optimizers import SPSA
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_algorithms.optimizers import COBYLA


from qiskit_nature.second_q.drivers import PySCFDriver
# from qiskit_nature.second_q.problems import ElectronicStructureProblem
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.second_q.circuit.library import HartreeFock
from qiskit_algorithms.minimum_eigensolvers import NumPyMinimumEigensolver

exact_solver = NumPyMinimumEigensolver()

estimator = Estimator()
mapper = ParityMapper()

# -------------------------
# H2 potential energy curve
# -------------------------
distances = np.linspace(0.5, 2.5, 40)
energies = []
energies_class = []


for R in distances:

    driver = PySCFDriver(
        atom=f"H 0 0 0; H 0 0 {R}",
        basis="sto3g", # 6-31G
        charge=0,
        spin=0,
    )

    problem = driver.run()
    
    # Map fermionic Hamiltonian → qubit Hamiltonian
    second_q_ops = problem.second_q_ops()[0]
    qubit_hamiltonian = mapper.map(second_q_ops)

   # Create Hartree-Fock initial state (specific to this Hamiltonian)
    hf_state = HartreeFock(
        num_spatial_orbitals=problem.num_spatial_orbitals,
        num_particles=problem.num_particles,
        qubit_mapper=mapper
    )

    ansatz = n_local(
        num_qubits=qubit_hamiltonian.num_qubits, # number of qubits in your mapped Hamiltonian
        rotation_blocks=["ry"], #type of rotations (["ry","rz"] is more expressive than just "ry")
        entanglement_blocks="cz", # type of entanglement (e.g., "cz" or "cx")
        entanglement="linear",  # linear instead of full
        reps=1
    )


    optimizer = SPSA (maxiter=1000) #COBYLA

    vqe = VQE(ansatz=ansatz, optimizer=optimizer, estimator=estimator)

    result = vqe.compute_minimum_eigenvalue(qubit_hamiltonian)

    electronic_energy = np.real(result.eigenvalue)
    nuclear_repulsion = problem.nuclear_repulsion_energy
    total_energy = electronic_energy + nuclear_repulsion
    energies.append(total_energy)
    
    print(f"R = {R:.2f} Å  ->  Energy= {total_energy:.6f} Hartree")


    # Classical (exact)
    exact_result = exact_solver.compute_minimum_eigenvalue(qubit_hamiltonian)
    energies_class.append(np.real(exact_result.eigenvalue + nuclear_repulsion))

print(qubit_hamiltonian)
ansatz.draw("mpl", style="iqp")
# -------------------------
# Plot
# -------------------------
plt.plot(distances, energies, "o-", label="VQE")
plt.plot(distances, energies_class, "--", label="Exact (classical)")
plt.xlabel("H–H distance (Å)")
plt.ylabel("Energy (Hartree)")
plt.title("H₂ potential energy curve (Modern VQE)")
plt.grid(True)

plt.show()


In [ ]:
ansatz.draw("mpl", style="iqp")